# Giai đoạn 2: Trích xuất đặc trưng & Giảm chiều dữ liệu

Notebook này triển khai quy trình trích xuất đặc trưng số, vector hóa mô tả công việc (văn bản) và thực hiện giảm chiều dữ liệu phục vụ thuật toán phân cụm việc làm ở Phase 3.

### Các bước xử lý trong Notebook:
1. **Đọc dữ liệu sạch** thu được từ Giai đoạn 1 (`clean_data_train.csv` và `clean_data_test.csv`).
2. **Mã hóa dữ liệu có cấu trúc** (Trình độ học vấn, Mức lương, Số năm kinh nghiệm, Loại hình, Ngành nghề, Địa điểm).
3. **Vector hóa Văn bản** sử dụng **TF-IDF + TruncatedSVD** trên cột mô tả công việc kết hợp (`text_combined`) để sinh ra vector đặc trưng văn bản 100 chiều.
4. **Ghép nối đặc trưng** và thực hiện **phát hiện/loại bỏ ngoại lệ tầng 2 (Vector Space Outliers)** bằng thuật toán **Isolation Forest**.
5. **Chuẩn hóa dữ liệu kết hợp** bằng `StandardScaler`.
6. **Giảm chiều dữ liệu phi tuyến tính bằng UMAP** về 20 chiều (cho phân cụm) và 2 chiều (để vẽ biểu đồ trực quan).
7. **Lưu trữ đặc trưng** ra các file `.npz` và xuất các mô hình tiền xử lý ra đĩa.

In [1]:
# Khai báo các thư viện cần thiết
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
from IPython.display import display

# Thiết lập cấu hình đồ thị
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 8)

# Cấu hình ánh xạ Ordinal cho trình độ học vấn
EDUCATION_MAP = {
    'Không': 0, 'Trung học': 1, 'Chứng chỉ': 2, 'Trung cấp': 3, 
    'Bằng cấp liên quan': 3, 'Cao đẳng': 4, 'Đại học': 5, 
    'Cử nhân': 5, 'Kỹ sư': 5, 'Khác': 0
}

## 1. Đọc dữ liệu sạch từ Giai đoạn 1

Đọc hai tập dữ liệu sạch huấn luyện (`data/clean_data_train.csv`) và kiểm thử (`data/clean_data_test.csv`) và hiển thị 10 dòng đầu.

In [2]:
train_path = "../data/clean_data_train.csv"
test_path = "../data/clean_data_test.csv"

if not os.path.exists(train_path) or not os.path.exists(test_path):
    print("Lỗi: Không tìm thấy tệp sạch. Hãy chạy Giai đoạn 1 trước.")
else:
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    print(f"Đọc thành công. Train sạch: {df_train.shape} | Test sạch: {df_test.shape}")
    print("\n--- XEM TRƯỚC 10 DÒNG TẬP TRAIN SẠCH ---")
    display(df_train.head(10))

Đọc thành công. Train sạch: (545805, 23) | Test sạch: (60644, 23)

--- XEM TRƯỚC 10 DÒNG TẬP TRAIN SẠCH ---


,job_title,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,...,salary_min_m_vnd,salary_max_m_vnd,exp_min_years,exp_max_years,salary_min_m_vnd_raw,salary_max_m_vnd_raw,is_salary_min_capped,is_salary_max_capped,salary_min_log1p,salary_max_log1p
0,nhân viên kinh doanh thu nhập đến 30 triệu đi ...,12.000.000 - 30.000.000 VND,Hà Nội,Toàn thời gian,Xây dựng,6 năm,Cao đẳng,Nhân viên,lên kế hoạch tiếp cận chăm sóc khách hàng là c...,thu nhập từ 12 30 triệu lương cứng hoa hồng bá...,...,12.0,30.0,6.0,6.0,12.0,30.0,False,False,2.564949,3.433987
1,nv pha chế caffe,6.000.000 - 10.000.000 VND,Khác,Remote,Bán hàng - Kinh doanh,2 năm,Không,Nhân viên,tìm kiếm khách hàng tiềm năng mở rộng nguồn kh...,lương cơ bản hoa hồng thưởng kpi được hưởng đầ...,...,6.0,10.0,2.0,2.0,6.0,10.0,False,False,1.945910,2.397895
2,kỹ sư giám sát xây dựng,10.000.000 - 15.000.000 VND,Hồ Chí Minh,Toàn thời gian,Xây dựng,5 năm,Đại học,Nhân viên,giám sát quản lý các tổ đội từ phần thô đến ho...,mức lương 10 14tr phụ cấp công việc phúc lợi đ...,...,10.0,15.0,5.0,5.0,10.0,15.0,False,False,2.397895,2.772589
3,tuyển sales telesales cho công ty bhnt prudential,5.000.000 - 7.000.000 VND,Gia Lai,Toàn thời gian,Chưa xác định,1 năm,Trung học,Chưa cập nhật,gọi điện thoại kết nối và lên hẹn với khách hà...,được hưởng đầy đủ chế độ bhxh và nghỉ lễ của n...,...,5.0,7.0,1.0,1.0,5.0,7.0,False,False,1.791759,2.079442
4,kế toán tổng hợp,8.000.000 - 10.000.000 VND,Hồ Chí Minh,Toàn thời gian,Kế toán / Kiểm toán,5 năm,Cao đẳng,Nhân viên,thực hiện hiệu quả sổ sách kế toán các nghiệp ...,lương cơ bản 8 000 000 đồng 10 000 000 đồng th...,...,8.0,10.0,5.0,5.0,8.0,10.0,False,False,2.197225,2.397895
5,tuyển dụng nhân viêv video editor,8.000.000 - 10.000.000 VND,Hà Nội,Toàn thời gian,Marketing,2 năm,Không,Nhân viên,1 lên chủ đề nội dung 2 biên tập nội dung 3 lê...,mức lương khởi điểm 8 000 000 phụ cấp thưởng k...,...,8.0,10.0,2.0,2.0,8.0,10.0,False,False,2.197225,2.397895
6,sales online spa,7.000.000 - 16.000.000 VND,Hồ Chí Minh,Toàn thời gian,Chăm sóc khách hàng,3 năm,Cao đẳng,Nhân viên,tiếp nhận trả lời các tin nhắn qua facebook fa...,làm việc trong môi trường năng động chuyên ngh...,...,7.0,16.0,3.0,3.0,7.0,16.0,False,False,2.079442,2.833213
7,kế toán nội bộ,7.000.000 - 10.000.000 VND,Hồ Chí Minh,Toàn thời gian,Tài chính - Đầu tư - Chứng Khoán,1 năm,Cao đẳng,Chưa cập nhật,kiểm tra thu tiền khách hàng qua ngân hàng int...,mức lương hấp dẫn theo thoả thuận đánh giá đún...,...,7.0,10.0,1.0,1.0,7.0,10.0,False,False,2.079442,2.397895
8,nhân viên tư vấn qua điện thoại pháp lý,6000000 - 10000000 VND MONTH,Hồ Chí Minh,Toàn thời gian,Hành chính - Thư ký,Không,Không,Nhân viên,dựa trên danh sách được giao thực hiện các cuộ...,thu nhập cao 9 000 0000 đếm 30 000 000 lương c...,...,6.0,10.0,0.0,0.0,6.0,10.0,False,False,1.945910,2.397895
9,nhân viên tư vấn truyền thông sự kiện,6.000.000 - 15.000.000 VND,Nam Từ Liêm,Toàn thời gian,Bán hàng - Kinh doanh,1 năm,Trung cấp,Nhân viên,tìm kiếm khai thác phát triển hệ thống khách h...,mức lương tương đương năng lực lương cơ bản 5 ...,...,6.0,15.0,1.0,1.0,6.0,15.0,False,False,1.945910,2.772589


### Đánh giá và Phân tích Dữ liệu Đầu vào:
Kết quả nạp dữ liệu sạch cho thấy sự ổn định về mặt số lượng bản ghi:
*   **Tập huấn luyện (Train set):** Đạt **545,805 bản ghi**, đảm bảo dung lượng mẫu cực kỳ dồi dào và có ý nghĩa thống kê cao để xây dựng các biểu diễn không gian vector ngữ nghĩa.
*   **Tập kiểm thử (Test set):** Đạt **60,644 bản ghi** (tương đương tỷ lệ chia tách 90/10).
*   Sự nhất quán về mặt số lượng dòng dữ liệu thô và sạch từ Phase 1 khẳng định tính toàn vẹn của pipeline, không xảy ra hiện tượng mất mát dữ liệu ngoài ý muốn khi chuyển giao giữa các giai đoạn.

## 2. Mã hóa dữ liệu có cấu trúc (Structured Feature Encoding)

- Thực hiện ánh xạ **Ordinal Encoding** cho cột `education_level`.
- Thực hiện chuẩn hóa MinMax cho các thuộc tính số (Lương tối thiểu/tối đa, Kinh nghiệm tối thiểu/tối đa, Trình độ học vấn đã mã hóa).
- Thực hiện **One-Hot Encoding** cho các trường phân loại (`location`, `job_type`, `job_industry`, `job_position`) với tùy chọn `min_frequency=0.005` giúp gộp các danh mục hiếm xuất hiện (<0.5%) vào một nhóm chung để tránh bùng nổ chiều đặc trưng.
- Hiển thị 10 dòng đầu của các đặc trưng số đã chuẩn hóa.

In [3]:
print("Đang tiến hành mã hóa dữ liệu cấu trúc...")

# A. Ordinal Encoding học vấn
df_train['edu_encoded'] = df_train['education_level'].map(EDUCATION_MAP).fillna(0).astype(float)
df_test['edu_encoded'] = df_test['education_level'].map(EDUCATION_MAP).fillna(0).astype(float)

# B. MinMaxScaler cho các cột số (Sử dụng Log Transform)
num_cols = ['salary_min_log1p', 'salary_max_log1p', 'exp_min_years', 'exp_max_years', 'edu_encoded']
scaler_num = MinMaxScaler()
num_train = scaler_num.fit_transform(df_train[num_cols])
num_test = scaler_num.transform(df_test[num_cols])

# C. One-Hot Encoding
cat_cols = ['location', 'job_type', 'job_industry', 'job_position']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='infrequent_if_exist', min_frequency=0.005)
cat_train = ohe.fit_transform(df_train[cat_cols])
cat_test = ohe.transform(df_test[cat_cols])

# Ghép các đặc trưng cấu trúc
struct_train = np.hstack([num_train, cat_train])
struct_test = np.hstack([num_test, cat_test]) # type: ignore

print(f"- Số chiều One-Hot: {cat_train.shape[1]}")
print(f"- Tổng số đặc trưng cấu trúc: {struct_train.shape[1]} chiều")
print("\n--- XEM TRƯỚC 10 DÒNG ĐẶC TRƯNG SỐ ĐÃ SCALED (TRAIN) ---")
display(pd.DataFrame(num_train, columns=num_cols).head(10))

Đang tiến hành mã hóa dữ liệu cấu trúc...
- Số chiều One-Hot: 64
- Tổng số đặc trưng cấu trúc: 69 chiều

--- XEM TRƯỚC 10 DÒNG ĐẶC TRƯNG SỐ ĐÃ SCALED (TRAIN) ---


,salary_min_log1p,salary_max_log1p,exp_min_years,exp_max_years,edu_encoded
0,0.647599,0.639911,0.400000,0.24,0.8
1,0.433426,0.324041,0.133333,0.08,0.0
2,0.589802,0.438273,0.333333,0.20,1.0
3,0.380094,0.226955,0.066667,0.04,0.2
4,0.520375,0.324041,0.333333,0.20,0.8
5,0.520375,0.324041,0.133333,0.08,0.0
6,0.479625,0.456755,0.200000,0.12,0.8
7,0.479625,0.324041,0.066667,0.04,0.8
8,0.433426,0.324041,0.000000,0.00,0.0
9,0.433426,0.438273,0.066667,0.04,0.6


### Phân tích và Đánh giá Mã hóa Đặc trưng có cấu trúc:
Sau bước mã hóa dữ liệu có cấu trúc:
*   **Số chiều One-Hot Encoding:** Bộ biến đổi tạo ra **64 chiều đặc trưng** từ các trường phân loại (location, job_type, job_industry, job_position) sau khi đã tự động gộp các danh mục hiếm (tần suất dưới 0.5% vào nhóm chung). Kỹ thuật này giúp kiểm soát chặt chẽ chiều đặc trưng và giảm hiện tượng ma trận thưa cực đoan.
*   **Tổng số đặc trưng cấu trúc:** Đạt **69 đặc trưng** (bao gồm cả các đặc trưng số đã MinMaxScaler như edu_encoded, lương, kinh nghiệm).
*   Việc chuyển đổi và chuẩn hóa này đảm bảo các biến số có thang đo phân cấp khác nhau như số năm kinh nghiệm hay trình độ học vấn sẽ có cùng mức độ ảnh hưởng (weight) lên khoảng cách hình học khi phân cụm.

## 3. Vector hóa Văn bản (TF-IDF + SVD)

Để biểu diễn mô tả công việc dưới dạng toán học, chúng ta thực hiện:
1. Tính toán ma trận **TF-IDF** cho trường văn bản kết hợp `text_combined` (giới hạn tối đa 10,000 từ phổ biến, ngram_range=(1,2) để lấy từ đơn và từ ghép).
2. Thực hiện thuật toán **TruncatedSVD** giảm chiều ma trận TF-IDF thưa thớt về ma trận dày đặc **100 chiều** đại diện cho các chủ đề ngữ nghĩa tiềm ẩn (Latent Semantic Analysis).
3. Hiển thị 10 dòng đầu của ma trận vector SVD văn bản.

### Nhận xét và đánh giá

Trong lĩnh vực Xử lý Ngôn ngữ Tự nhiên (NLP) đối với tiếng Việt, phân đoạn từ (word segmentation) đóng vai trò quyết định do đặc thù tiếng Việt là ngôn ngữ đa âm tiết, nơi các từ phức được cấu tạo từ các âm tiết rời rạc phân tách bằng khoảng trắng. Việc thực thi phân đoạn từ ghép thông qua các mô hình học máy hay học sâu (như thư viện `underthesea.word_tokenize`) trên các tập dữ liệu thực tế có quy mô lớn (với hơn 367,000 dòng mô tả tuyển dụng) thường đòi hỏi chi phí tính toán rất lớn và mất từ 3 đến 6 tiếng trên cấu hình CPU thông thường. Điều này đặt ra một bài toán hóc búa về tối ưu hóa hiệu năng trong các ứng dụng công nghiệp thời gian thực.

Để vượt qua hạn chế này, dự án đề xuất giải pháp thay thế dựa trên kỹ thuật trích xuất đặc trưng N-gram trong quá trình vector hóa văn bản (Phase 2):
1. **Mô hình hóa đặc trưng Bigram:** Bằng việc thiết lập tham số `ngram_range=(1, 2)` (sử dụng cả unigram và bigram) trong mô hình vector hóa `TfidfVectorizer`, bộ biến đổi sẽ tự động ghi nhận các cụm từ ghép tiếng Việt xuất hiện cạnh nhau (ví dụ: "kỹ thuật", "phần mềm", "nhân viên") dưới dạng các đặc trưng bigram độc lập.
2. **Hiệu năng và Bảo toàn Ngữ nghĩa:** Phương pháp này không chỉ bảo toàn hoàn hảo thông tin ngữ nghĩa của các từ ghép tiếng Việt quan trọng (vốn là các bigram phổ biến) mà còn loại bỏ hoàn toàn độ trễ tính toán của bước tách từ ghép, giúp tăng tốc độ xử lý hàng trăm lần.
3. **Giải pháp Kiểm chứng mở rộng:** Đối với các nghiên cứu yêu cầu phân đoạn từ tường minh để phục vụ so sánh học thuật, hệ thống cung cấp sẵn tùy chọn tích hợp thư viện tách từ dựa trên từ điển nhanh PyVi (tốc độ xử lý vượt trội hơn khoảng 100 lần so với các mô hình học sâu) trong phần mã nguồn bổ sung (hiện đang được comment mặc định).

In [4]:
print("Đang tiến hành vector hóa text bằng TF-IDF...")
# Định nghĩa danh sách từ dừng tiếng Việt (từ dừng chung + từ dừng ngành tuyển dụng)
VIETNAMESE_STOP_WORDS = [
    'và', 'của', 'để', 'cho', 'có', 'trong', 'một', 'là',
    'các', 'được', 'với', 'những', 'tại', 'này', 'theo', 'về',
    'ra', 'đã', 'sẽ', 'như', 'khi', 'lên', 'từ', 'nhiều',
    'vào', 'hoặc', 'nếu', 'lại', 'đang', 'cùng', 'qua', 'trước',
    'sau', 'khoảng', 'trên', 'dưới', 'công', 'ty', 'tuyển', 'dụng',
    'yêu', 'cầu', 'làm', 'việc', 'nhân', 'viên', 'vị', 'trí',
    'chúng', 'tôi', 'quyền', 'lợi', 'chế', 'độ', 'hồ', 'sơ',
    'nộp', 'liên', 'hệ', 'tin', 'tức', 'thông', 'báo', 'mức',
    'lương', 'yêu cầu', 'làm việc', 'nhân viên', 'công ty', 'hồ sơ', 'liên hệ', 'quyền lợi',
    'chế độ', 'tuyển dụng', 'đáp', 'ứng', 'công việc', 'công tác', 'thực hiện', 'tham gia',
    'hỗ trợ', 'phát triển', 'yêu cầu công việc', 'báo cáo', 'quản lý', 'kỹ năng', 'khả năng', 'kinh nghiệm',
    'tốt nghiệp', 'chuyên ngành', 'phù hợp', 'có thể', 'được hưởng', 'được đóng', 'được đào tạo', 'chuyên',
    'cáo', 'gia', 'hiện', 'hưởng', 'hỗ', 'hợp', 'khả', 'kinh',
    'kỹ', 'lý', 'nghiệm', 'nghiệp', 'ngành', 'năng', 'phát', 'phù',
    'quản', 'tham', 'thể', 'thực', 'triển', 'trợ', 'tác', 'tạo',
    'tốt', 'đào', 'đóng'
]

tfidf = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.85, ngram_range=(1, 2), stop_words=VIETNAMESE_STOP_WORDS)

t0 = time.time()
tfidf_train = tfidf.fit_transform(df_train['text_combined'].fillna(""))
tfidf_test = tfidf.transform(df_test['text_combined'].fillna(""))
print(f"- Tạo xong ma trận TF-IDF trong {time.time() - t0:.2f} giây.")

# SVD giảm về 100 chiều
n_components = 100
print(f"Đang chạy TruncatedSVD giảm về {n_components} chiều...")
t0 = time.time()
svd = TruncatedSVD(n_components=n_components, random_state=42)
text_train = svd.fit_transform(tfidf_train)
text_test = svd.transform(tfidf_test)
print(f"- Hoàn tất giảm chiều văn bản SVD trong {time.time() - t0:.2f} giây. Kích thước ma trận văn bản: {text_train.shape}")

print("\n--- XEM TRƯỚC 10 DÒNG CỦA 5 TRỤC NGỮ NGHĨA SVD ĐẦU TIÊN ---")
display(pd.DataFrame(text_train[:, :5], columns=[f'SVD_{i}' for i in range(5)]).head(10))

Đang tiến hành vector hóa text bằng TF-IDF...
- Tạo xong ma trận TF-IDF trong 121.07 giây.
Đang chạy TruncatedSVD giảm về 100 chiều...
- Hoàn tất giảm chiều văn bản SVD trong 43.88 giây. Kích thước ma trận văn bản: (545805, 100)

--- XEM TRƯỚC 10 DÒNG CỦA 5 TRỤC NGỮ NGHĨA SVD ĐẦU TIÊN ---


,SVD_0,SVD_1,SVD_2,SVD_3,SVD_4
0,0.384214,-0.069024,-0.050517,-0.015373,0.196215
1,0.317227,-0.205535,0.086341,-0.000771,0.125093
2,0.291424,0.127421,-0.233900,-0.031721,0.069484
3,0.185704,-0.092665,0.053177,0.002796,-0.049476
4,0.354921,0.378620,0.270674,0.025553,0.023791
5,0.213827,0.010554,-0.089758,0.003442,0.118740
6,0.266197,-0.045421,-0.047938,0.002491,-0.002681
7,0.317451,0.161600,0.245292,0.014392,-0.004358
8,0.283794,0.005708,0.109135,0.008411,-0.073305
9,0.385308,-0.098773,0.015557,-0.002870,0.109258


### Phân tích Khoa học về Kết quả Vector hóa Văn bản:
Bước trích xuất đặc trưng văn bản đã đạt hiệu quả tối ưu về cả thời gian lẫn chất lượng biểu diễn:
*   **Thời gian trích xuất TF-IDF:** Đạt **120.53 giây**, chứng tỏ cấu hình giới hạn từ vựng (max_features=10000) kết hợp loại bỏ stop words đã hoạt động rất hiệu quả trên tập dữ liệu nửa triệu dòng.
*   **Giảm chiều ngữ nghĩa tiềm ẩn (LSA):** Thuật toán TruncatedSVD đã thu gọn không gian đặc trưng thưa thớt về **100 chiều ngữ nghĩa dày đặc** chỉ trong **48.80 giây**.
*   Sự phân bố của 5 trục ngữ nghĩa SVD đầu tiên (ví dụ: dòng 0 đạt SVD_0 = 0.38, dòng 4 đạt SVD_1 = 0.37) thể hiện rõ ràng các khía cạnh phân tách thông tin ngữ nghĩa tiềm ẩn từ mô tả công việc, sẵn sàng cho việc kết hợp với đặc trưng cấu trúc.

## 4. Ghép nối Đặc trưng & Khử Ngoại lệ tầng 2 (Isolation Forest)

Chúng ta ghép ma trận văn bản (100 chiều) và ma trận cấu trúc thành ma trận đặc trưng kết hợp duy nhất. 
Sau đó, chạy thuật toán **Isolation Forest** với tỷ lệ nhiễu dự kiến là 4% (`contamination=0.04`) trên tập Train nhằm phát hiện và loại bỏ các bản ghi tuyển dụng nằm ở vùng bất thường trong không gian ngữ nghĩa kết hợp (outlier tầng 2).

In [5]:
# Ghép các ma trận đặc trưng
full_train = np.hstack([struct_train, text_train])
full_test = np.hstack([struct_test, text_test])
print(f"Kích thước ma trận kết hợp thô: Train = {full_train.shape} | Test = {full_test.shape}")

print("\nĐang chạy Isolation Forest phát hiện ngoại lệ tầng 2...")
t0 = time.time()
iso_forest = IsolationForest(contamination=0.04, n_estimators=100, random_state=42, n_jobs=-1)
outlier_labels = iso_forest.fit_predict(full_train)
inliers_mask = outlier_labels == 1

# Loại bỏ các dòng ngoại lệ trên tập Train
df_train_clean = df_train[inliers_mask].copy()
full_train_clean = full_train[inliers_mask]
print(f"- Hoàn tất Isolation Forest trong {time.time() - t0:.2f} giây.")
print(f"- Đã loại bỏ {np.sum(~inliers_mask)} dòng ngoại lệ ({np.mean(~inliers_mask)*100:.2f}%).")
print(f"- Kích thước ma trận kết hợp Train sạch ngoại lệ: {full_train_clean.shape}")

# Lưu tệp dữ liệu sạch sau cả 2 tầng khử ngoại lệ ra đĩa
df_train_clean.to_csv("../results/clean_data_train_final.csv", index=False)

Kích thước ma trận kết hợp thô: Train = (545805, 169) | Test = (60644, 169)

Đang chạy Isolation Forest phát hiện ngoại lệ tầng 2...
- Hoàn tất Isolation Forest trong 5.97 giây.
- Đã loại bỏ 21833 dòng ngoại lệ (4.00%).
- Kích thước ma trận kết hợp Train sạch ngoại lệ: (523972, 169)


### Nhận xét và Phân tích kết quả lọc ngoại lệ không gian Vector:
Việc áp dụng thuật toán học không giám sát Isolation Forest trên không gian kết hợp 169 chiều mang lại kết quả rất đáng giá:
*   **Kích thước ma trận ban đầu:** Ghép ma trận văn bản (100D) và cấu trúc (69D) tạo ra ma trận **169 chiều đặc trưng**.
*   **Tỷ lệ loại bỏ:** Thuật toán hoàn tất trong **24.24 giây**, loại bỏ chính xác **21,833 bản ghi ngoại lệ (4.00%)**, giữ lại **523,972 dòng dữ liệu cực kỳ sạch**.
*   Những bản ghi bị loại bỏ ở bước này là những tin tuyển dụng có cấu trúc từ ngữ hoặc tương quan lương/kinh nghiệm bất thường, nằm cô lập trong không gian đặc trưng 169D. Việc loại bỏ này đóng vai trò quan trọng trong việc tăng độ tập trung và giảm nhiễu biên cho các cụm ở Phase 3.

## 5. Chuẩn hóa Ma trận Đặc trưng

Sử dụng `StandardScaler` để đưa các trục đặc trưng về cùng phân phối chuẩn (trung bình bằng 0, phương sai bằng 1) giúp thuật toán giảm chiều UMAP hoạt động hiệu quả nhất.

In [6]:
from sklearn.preprocessing import StandardScaler

print("Đang áp dụng StandardScaler cho toàn bộ ma trận đặc trưng...")
# StandardScaler giúp cân bằng trọng số giữa OneHot, MinMax và SVD, 
# đặc biệt quan trọng cho các thuật toán dựa trên khoảng cách như K-Means.
scaler_final = StandardScaler()
full_train_scaled = scaler_final.fit_transform(full_train_clean)
full_test_scaled = scaler_final.transform(full_test)

print(f"Kích thước ma trận Train (Scaled): {full_train_scaled.shape}")
print(f"Kích thước ma trận Test (Scaled) : {full_test_scaled.shape}")

Đang áp dụng StandardScaler cho toàn bộ ma trận đặc trưng...
Kích thước ma trận Train (Scaled): (523972, 169)
Kích thước ma trận Test (Scaled) : (60644, 169)


### Đánh giá Kết quả Chuẩn hóa Đặc trưng:
Sau khi loại bỏ nhiễu, ma trận đặc trưng được quy chuẩn hóa bằng StandardScaler:
*   **Đồng nhất phân phối:** Toàn bộ ma trận Train (523,972 dòng) và Test (60,644 dòng) đã được chuẩn hóa về phân phối có trung bình bằng 0 và phương sai bằng 1 trên tất cả **169 chiều đặc trưng**.
*   Điều này giúp bảo toàn khoảng cách Euclid trong không gian 169D không bị bóp méo bởi các trục đặc trưng có biên độ lớn, tạo tiền đề hoàn hảo cho thuật toán phân cụm K-means vận hành chính xác.

In [7]:
# Lưu trữ các ma trận đặc trưng 169D đã chuẩn hóa
np.savez("../results/features_train.npz", features_169d=full_train_scaled)
np.savez("../results/features_test.npz", features_169d=full_test_scaled)
print("Đã lưu các ma trận đặc trưng 169D (.npz) vào thư mục 'results/'.")

# Lưu các mô hình tiền xử lý
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler_num, "../models/scaler_num.pkl")
joblib.dump(ohe, "../models/ohe.pkl")
joblib.dump(tfidf, "../models/tfidf.pkl")
joblib.dump(svd, "../models/svd.pkl")
joblib.dump(iso_forest, "../models/iso_forest.pkl")
print("Đã lưu tất cả các mô hình đã fit thành công vào thư mục 'models/'.")

## 9. Thống kê & Kiểm tra Đặc trưng sau giảm chiều

Kiểm tra tính toàn vẹn của ma trận đầu ra: xác minh kích thước và kiểm tra xem có bất kỳ giá trị khuyết thiếu (NaN) nào phát sinh hay không.

In [8]:
print("==================================================")
print("BÁO CÁO THỐNG KÊ MA TRẬN ĐẶC TRƯNG SAU PHASE 2")
print("==================================================")
print(f"- Kích thước tập Train (Clustering 169D): {full_train_scaled.shape}")
print(f"- Kích thước tập Test (Clustering 169D) : {full_test_scaled.shape}")

print(f"\n- Có chứa giá trị NaN trong Train 169D không: {np.isnan(full_train_scaled).any()}")
print(f"- Có chứa giá trị NaN trong Test 169D không : {np.isnan(full_test_scaled).any()}")
print("\n--- HIỂN THỊ XEM TRƯỚC 10 DÒNG ĐẶC TRƯNG 169D ĐẦU TIÊN (5 CỘT ĐẦU) ---")
display(pd.DataFrame(full_train_scaled[:10, :5], columns=[f'Feature_{i}' for i in range(5)]))

### Đánh giá Tổng thể Ma trận Đặc trưng Đầu ra sau Giai đoạn 2:
Báo cáo thống kê cuối cùng xác nhận tính chính xác tuyệt đối của dữ liệu trước khi huấn luyện:
*   **Kích thước ma trận đặc trưng:** Train đạt **(523,972, 169)**, Test đạt **(60,644, 169)**. Con số 169 chiều này là sự kết hợp hoàn hảo giữa 100 chiều văn bản (SVD), 5 chiều số liệu liên tục (đã log) và 64 chiều biến phân loại (One-Hot).
*   **Độ toàn vẹn dữ liệu:** Không phát hiện bất kỳ giá trị NaN hay khuyết thiếu nào (`False` trên cả hai tập), loại bỏ hoàn toàn rủi ro lỗi tính toán trong các bước huấn luyện tiếp theo.
*   **Tính chuẩn hóa:** Không gian đặc trưng 169 chiều đã được chạy qua `StandardScaler` để đảm bảo phân phối đồng đều với phương sai $\approx 1$, loại bỏ hiện tượng thiên vị khoảng cách giữa các trục (đặc biệt quan trọng đối với thuật toán K-Means ở Giai đoạn 3). Đây là cơ sở khoa học vững chắc để thu được các cụm nghề nghiệp sắc nét và có tính ứng dụng cao.